# Predictive vs predicted train-test regression matrix

This notebook runs chronological train-test regressions for every available predictive time series against every available predicted time series.

The aim is not just to fit associations, but to test whether each candidate predictive series improves held-out prediction relative to a simple training-mean baseline.

## Dataset roles

Automatically handled predictive series:

- Google Trends 1-year files in `Google_trends_v2/1y_data/time_series_GB*`
- UKHSA charts classified as NHS-call series
- processed wastewater long-format data, if `data/processed/wastewater_long.{parquet,csv}` exists

Automatically handled predicted series:

- UKHSA charts classified as GP/admission series

Clinical raw NHS England datasets can be added to this framework once their columns have been mapped into the same canonical series format.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt

from wastewater.io import find_repo_root
from wastewater.regression_matrix import (
    build_available_series,
    summarise_available_series,
    expected_family_status,
    run_pairwise_train_test_matrix,
)

ROOT = find_repo_root(ROOT)
PROCESSED = ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)
ROOT

## 1. Build canonical series catalogue

Everything is converted into a simple long format with columns like `date`, `value`, `series_id`, `role`, and `dataset_family`.

In [ ]:
series = build_available_series(ROOT)
print(f'Canonical observations: {len(series):,}')
display(series.head())

## 2. Check which expected families are available

In [ ]:
family_status = expected_family_status(ROOT, series)
display(family_status)

## 3. Inspect available series

In [ ]:
series_summary = summarise_available_series(series)
display(series_summary)

if not series_summary.empty:
    display(series_summary.groupby(['role', 'dataset_family'])['series_id'].nunique().reset_index(name='n_series'))

## 4. Run pairwise chronological train-test regressions

For each predictive/predicted pair:

1. aggregate both series to the chosen frequency,
2. create lagged predictor values,
3. split chronologically into train and test,
4. standardise using the training window,
5. fit OLS on the training rows,
6. predict the held-out test rows,
7. compare against a training-mean baseline.

In [ ]:
FREQ = 'W'
LAGS = [0, 1, 2, 3, 4]
TRAIN_FRACTION = 0.8
MIN_TEST_SIZE = 4

results, predictions = run_pairwise_train_test_matrix(
    series,
    freq=FREQ,
    lags=LAGS,
    train_fraction=TRAIN_FRACTION,
    min_test_size=MIN_TEST_SIZE,
    aggregation='mean',
)

display(results)

results_path = PROCESSED / 'predictive_vs_predicted_train_test_results.csv'
predictions_path = PROCESSED / 'predictive_vs_predicted_train_test_predictions.csv'
results.to_csv(results_path, index=False)
predictions.to_csv(predictions_path, index=False)
print(f'Saved results to {results_path.relative_to(ROOT)}')
print(f'Saved predictions to {predictions_path.relative_to(ROOT)}')

## 5. Rank predictive performance

The key column is `mse_skill_vs_train_mean`. Positive values mean the model beats the training-mean baseline on the held-out test period. Negative values mean it is worse than the baseline.

In [ ]:
ok = results[results['status'] == 'ok'].copy() if not results.empty else pd.DataFrame()

ranking_cols = [
    'predictor_family',
    'predictor_name',
    'target_family',
    'target_name',
    'n_train',
    'n_test',
    'rmse',
    'baseline_rmse',
    'mse_skill_vs_train_mean',
    'correlation',
    'r2',
]

if ok.empty:
    print('No successful regressions. Inspect the error rows in results.')
else:
    display(ok[ranking_cols].sort_values('mse_skill_vs_train_mean', ascending=False).head(30))

errors = results[results['status'] == 'error'] if not results.empty and 'status' in results.columns else pd.DataFrame()
if not errors.empty:
    print(f'Errored regressions: {len(errors)}')
    display(errors[['predictor_family', 'predictor_name', 'target_family', 'target_name', 'error']].head(20))

## 6. Plot the best held-out result

In [ ]:
if not ok.empty and not predictions.empty:
    best = ok.sort_values('mse_skill_vs_train_mean', ascending=False).iloc[0]
    pred = predictions[
        (predictions['predictor_id'] == best['predictor_id']) &
        (predictions['target_id'] == best['target_id'])
    ].copy()

    train = pred[pred['split'] == 'train']
    test = pred[pred['split'] == 'test']

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(train['period'], train['y_z'], label='Observed train', marker='o')
    ax.plot(train['period'], train['prediction'], label='Fitted train', marker='o')
    ax.plot(test['period'], test['y_z'], label='Observed test', marker='o')
    ax.plot(test['period'], test['prediction'], label='Predicted test', marker='o')
    ax.axvline(test['period'].min(), linestyle='--')
    ax.set_title(
        'Best held-out pair: '
        + str(best['predictor_name'])
        + ' → '
        + str(best['target_name'])
    )
    ax.set_xlabel('Period')
    ax.set_ylabel('Target, z-scored using train window')
    ax.legend()
    fig.autofmt_xdate()
    plt.show()

    display(best.to_frame('value'))
    display(test[['period', 'y_z', 'prediction', 'residual']])

## 7. Interpretation notes

- Positive `mse_skill_vs_train_mean` is the main first-pass signal of predictive value.
- `r2` on a very small test set can be unstable, so compare it with RMSE and the plotted held-out trajectory.
- If many predictors perform poorly, try fewer lags or monthly aggregation.
- If wastewater is absent, run `notebooks/01_wastewater_analysis.ipynb` first to create `data/processed/wastewater_long`.
- If NHS England raw clinical data should be included, add an explicit mapping from its raw columns to canonical time-series fields.